# CGRA & VC — 1st Assignment: Medieval Siege Camp
**Group members:** _name (student number)_, _name (student number)_

This notebook builds a small medieval scene in **WebGL 1.0** with our own vertex/fragment shaders and our own
procedural geometry (no geometry libraries). Every class inherits from the course class `CGRAobject`.

**How to run:** run all cells from top to bottom (*Cell → Run All*). Then **click on the canvas** (so it gets the
keyboard focus; while it has focus the Jupyter shortcuts are switched off) and use:

| Key | Action |
|---|---|
| `W` `S` `A` `D` | visitor camera: forward / backward / left / right (A.6) |
| `Q` `E` or `←` `→`, `↑` `↓` | turn left / right, look up / down |
| `V` | toggle visitor camera ↔ ride-along camera on the rear wagon (A.7) |
| `R` | wheels of the convoy: cubes → cylinders → spheres (A.2) |
| `G` | open / close the castle portcullis |
| `P` | pause / resume the convoy |

## Course libraries

In [ ]:
%%html
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/cgraobject.js"></script>
<script src='https://git.io/glm-js.min.js'></script>

## Shaders
The vertex shader receives the position and the colour of each vertex and one uniform matrix `MVP`.
For every object we compute on the CPU the product $MVP = P\,V\,M$ (projection × view × model), so the shader only
does one matrix–vector multiplication. The colour is passed to the fragment shader, which interpolates it.

In [ ]:
%%html
<script id="my-vertex-shader" type="x-shader/x-vertex">
precision mediump float;

attribute  vec3 in_Position;
attribute  vec3 in_Color;
uniform mat4 MVP;

varying  vec3 ex_Color;

void main(void) {
    gl_Position = MVP * vec4(in_Position.x, in_Position.y, in_Position.z, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="my-fragment-shader" type="x-shader/x-fragment">
precision mediump float;

varying  vec3 ex_Color;

void main(void) {
    gl_FragColor = vec4(ex_Color,1.0);
}
</script>

## Exercise A.1 — Basic primitives

All primitives are generated procedurally as lists of independent triangles (drawn with `gl.TRIANGLES`), centred at
the origin and inside the box $[-1,1]^3$. Placing them in the scene is then only a matter of scaling, rotating and
translating.

* **Cube** — 6 faces × 2 triangles × 3 vertices = **36 vertices**. The 8 corners are written explicitly and repeated
  for each face, so every face has its own vertices (needed for independent face attributes).
* **Disc** — $n = 20$ points on the circle $(\cos\alpha_k, 0, \sin\alpha_k)$, $\alpha_k = 2\pi k/n$; each consecutive
  pair is joined to the centre → $n$ triangles.
* **Cone** — the same base circle at $y=-1$; every consecutive pair is joined to the apex $(0,1,0)$. The base is closed
  with a disc.
* **Cylinder** — two circles at $y=\pm1$. Vertices $k, k{+}1$ of the bottom and top circles form a quad, split into
  two triangles. With `closed = true` the two caps (discs) are added; with `closed = false` we get an **open
  cylinder** (used for the barrels, whose lids are discs).
* **Sphere** (radius 1) — 12 latitude rings × 20 vertices. Ring $i$ is at elevation
  $\varphi_i = -\tfrac{\pi}{2} + \pi i/12$, with radius $\cos\varphi_i$ and height $\sin\varphi_i$. Neighbouring rings
  are joined like a cylinder; at the poles the radius is 0 and the strip degenerates into a cone.

**Colours.** Every constructor receives two colours $c_1, c_2$. For each vertex we compute
$t = (x - x_{min})/(x_{max}-x_{min})$ and use $c = c_1 + t\,(c_2 - c_1)$: a smooth horizontal gradient from $c_1$ (left)
to $c_2$ (right).

The class `Shape` (which extends `CGRAobject`) uploads the two arrays to GPU buffers exactly like the `triangle`
class of the course example; `Cube`, `Disc`, `Cone`, `Cylinder` and `Sphere` extend `Shape`.
Classes are stored on `window` so that the cell can be executed more than once.

In [ ]:
%%html
<script>
// =====================================================================
//  EXERCISE A.1 - Basic primitives
// ---------------------------------------------------------------------
//  Every shape is a list of independent triangles (3 floats per vertex).
//  All shapes live inside the box [-1,1]^3, centred at the origin, y up,
//  so later we only need scale/rotate/translate matrices to place them.
//  Classes are stored on `window` so this cell can be run again safely.
// =====================================================================

// ---------- small geometry helpers ----------

// Linear mix of two RGB colours, t in [0,1]
function mixColor(c1, c2, t) {
  return [c1[0] + (c2[0] - c1[0]) * t,
          c1[1] + (c2[1] - c1[1]) * t,
          c1[2] + (c2[2] - c1[2]) * t];
}

// One colour per vertex: horizontal gradient along x.
// Left-most vertex gets c1, right-most vertex gets c2.
function gradientColors(vertices, c1, c2) {
  let minX = Infinity, maxX = -Infinity;
  for (let i = 0; i < vertices.length; i += 3) {
    minX = Math.min(minX, vertices[i]);
    maxX = Math.max(maxX, vertices[i]);
  }
  const span = (maxX - minX) || 1;
  const colors = [];
  for (let i = 0; i < vertices.length; i += 3)
    colors.push(...mixColor(c1, c2, (vertices[i] - minX) / span));
  return colors;
}

// Append one triangle (three [x,y,z] points) to a flat array
function tri(out, a, b, c) { out.push(...a, ...b, ...c); }

// Point k of n on a circle of radius r at height y (circle in the XZ plane)
function circlePoint(r, y, k, n) {
  const a = 2 * Math.PI * k / n;
  return [r * Math.cos(a), y, r * Math.sin(a)];
}

// ---------- vertex generators ----------

// CUBE: 6 faces x 2 triangles x 3 vertices = 36 vertices.
// The 8 corners are repeated so each face owns its own vertices.
function buildCube() {
  const p = [[-1,-1,-1], [ 1,-1,-1], [ 1, 1,-1], [-1, 1,-1],   // back  z=-1
             [-1,-1, 1], [ 1,-1, 1], [ 1, 1, 1], [-1, 1, 1]];  // front z=+1
  const faces = [[4,5,6,7], [1,0,3,2], [0,4,7,3],               // front, back, left
                 [5,1,2,6], [7,6,2,3], [0,1,5,4]];              // right, top, bottom
  const v = [];
  for (const [a, b, c, d] of faces) {        // each quad -> 2 triangles
    tri(v, p[a], p[b], p[c]);
    tri(v, p[a], p[c], p[d]);
  }
  return v;
}

// DISC (y=0): every pair of consecutive perimeter points + the centre
function buildDisc(n) {
  const v = [];
  for (let k = 0; k < n; k++)
    tri(v, [0, 0, 0], circlePoint(1, 0, k + 1, n), circlePoint(1, 0, k, n));
  return v;
}

// CONE: base circle at y=-1, apex at y=+1, base closed by a disc
function buildCone(n) {
  const v = [];
  for (let k = 0; k < n; k++) {
    const p0 = circlePoint(1, -1, k, n), p1 = circlePoint(1, -1, k + 1, n);
    tri(v, p0, [0, 1, 0], p1);               // side triangle to the apex
    tri(v, [0, -1, 0], p0, p1);              // base triangle
  }
  return v;
}

// CYLINDER: circles at y=-1 and y=+1. Each pair of consecutive bottom/top
// vertices forms a quad = 2 triangles. closed=true adds the two caps.
function buildCylinder(n, closed) {
  const v = [];
  for (let k = 0; k < n; k++) {
    const b0 = circlePoint(1, -1, k, n), b1 = circlePoint(1, -1, k + 1, n);
    const t0 = circlePoint(1,  1, k, n), t1 = circlePoint(1,  1, k + 1, n);
    tri(v, b0, t0, t1);
    tri(v, b0, t1, b1);
    if (closed) {
      tri(v, [0,  1, 0], t1, t0);            // top cap
      tri(v, [0, -1, 0], b0, b1);            // bottom cap
    }
  }
  return v;
}

// SPHERE (radius 1): rings at elevation phi have radius cos(phi) and
// height sin(phi). Neighbouring rings are joined like a cylinder; at the
// poles the radius is 0, so those strips become cones.
function buildSphere(stacks, slices) {
  const v = [];
  const ring = (i, k) => {
    const phi = -Math.PI / 2 + Math.PI * i / stacks;   // -90 .. +90 degrees
    return circlePoint(Math.cos(phi), Math.sin(phi), k, slices);
  };
  for (let i = 0; i < stacks; i++)
    for (let k = 0; k < slices; k++) {
      const a = ring(i, k), b = ring(i, k + 1), c = ring(i + 1, k), d = ring(i + 1, k + 1);
      tri(v, a, c, d);
      tri(v, a, d, b);
    }
  return v;
}

// ---------- classes (all inherit from CGRAobject) ----------

// Common base: uploads vertices + gradient colours to GPU buffers,
// exactly like the `triangle` class of 04Objects.ipynb.
window.Shape = class Shape extends CGRAobject {
  constructor(gl, vertices, c1, c2) {
    super(gl);
    this.numvertices = vertices.length / 3;
    const colors = gradientColors(vertices, c1, c2);

    this.vertexbuffer = this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vertexbuffer);
    this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(vertices), this.gl.STATIC_DRAW);

    this.colorbuffer = this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.colorbuffer);
    this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(colors), this.gl.STATIC_DRAW);
  }
};

window.Cube = class Cube extends Shape {
  constructor(gl, c1, c2) { super(gl, buildCube(), c1, c2); }
};
window.Disc = class Disc extends Shape {
  constructor(gl, c1, c2, n = 20) { super(gl, buildDisc(n), c1, c2); }
};
window.Cone = class Cone extends Shape {
  constructor(gl, c1, c2, n = 20) { super(gl, buildCone(n), c1, c2); }
};
window.Cylinder = class Cylinder extends Shape {          // closed = with caps
  constructor(gl, c1, c2, closed = true, n = 20) { super(gl, buildCylinder(n, closed), c1, c2); }
};
window.Sphere = class Sphere extends Shape {
  constructor(gl, c1, c2, stacks = 12, slices = 20) { super(gl, buildSphere(stacks, slices), c1, c2); }
};
</script>

## Section A.3 — Hierarchical (compound) objects

`Node3D` (also a `CGRAobject`) stores a **local matrix**, an optional primitive and a list of children. When drawing,

$$M_{world}(\text{node}) = M_{world}(\text{parent})\; M_{local}(\text{node})$$

so a transformation applied to a parent cascades to all its children (e.g. moving a wagon moves its wheels, load and
tow bar; rotating the trebuchet arm also moves the projectile and the counterweight).

All transformations studied in class are used:

* **translation** `T(x,y,z)` — positioning every part;
* **isometric scaling** `S(s,s,s)` — trees of different sizes, stone balls;
* **anisometric scaling** `S(sx,sy,sz)` — beams, walls, planks, wheels (cubes become boxes, spheres become ellipsoids…);
* **rotation** `R(angle, axis)` — cylinders lying along $x$ or $z$, roof planks, rotating wheels, blades and arm;
* **shear** `SH(k)` ($x' = x + k\,y$) — the leaning uprights of the trebuchet: top and bottom stay horizontal while the
  post leans.

`mul(A, B, C)` returns $A\,B\,C$; read right to left, the part is first scaled, then rotated, then translated.
The same cell also contains the convoy path (`lemniscate`) and the rotation from the tangent (`headingMatrix`),
explained in Exercise A.3 below.

In [ ]:
%%html
<script>
// =====================================================================
//  Hierarchical objects (section A.3) + transformation helpers
// ---------------------------------------------------------------------
//  A Node3D holds a LOCAL matrix, an optional shape and a list of
//  children. Its WORLD matrix is  parentWorld * local, so whatever we do
//  to a parent (move, rotate, scale) cascades down to all its children.
// =====================================================================

// Short names for the glm-js matrices (all angles in degrees)
function T(x, y, z)       { return glm.translate(glm.mat4(1), glm.vec3(x, y, z)); }
function S(x, y, z)       { return glm.scale(glm.mat4(1), glm.vec3(x, y, z)); }
function R(deg, x, y, z)  { return glm.rotate(glm.mat4(1), glm.radians(deg), glm.vec3(x, y, z)); }
function SH(kxy)          { const m = glm.mat4(1); m[1][0] = kxy; return m; }   // shear: x' = x + k*y

// Product of several matrices, left to right: mul(A,B,C) = A*B*C
function mul(...ms) { return ms.reduce((acc, m) => acc['*'](m), glm.mat4(1)); }

window.Node3D = class Node3D extends CGRAobject {
  constructor(gl, shape = null, local = glm.mat4(1)) {
    super(gl);
    this.part = shape;      // a Shape (Cube, Sphere...) or null for a pure group
    this.localM = local;    // transformation relative to the parent
    this.kids = [];
  }
  add(child) { this.kids.push(child); return child; }

  // Shortcut: add a primitive with a local transformation
  addPart(shape, ...mats) { return this.add(new Node3D(this.gl, shape, mul(...mats))); }

  // PV = projection*view, parent = world matrix of the parent node
  drawTree(PV, parent) {
    const world = parent['*'](this.localM);
    if (this.part) {
      // the course shader has a single uniform "MVP", so we send P*V*M
      this.part.setModelTransformation(PV['*'](world));
      this.part.drawit();
    }
    for (const k of this.kids) k.drawTree(PV, world);
  }
};

// =====================================================================
//  EXERCISE A.3 - Lemniscate of Bernoulli path
// =====================================================================
//  w = eps*t + theta.  Returns the position (x,y) and the tangent (dx,dy).
//  The road lies on the ground: path x -> world X, path y -> world Z.
function lemniscate(t, a, b, eps, theta) {
  const w = eps * t + theta;
  const s = Math.sin(w), c = Math.cos(w), D = 1 + s * s;
  return {
    x:  a * c / D,
    y:  b * s * c / D,
    dx: -a * eps * s * (2 + c * c) / (D * D),
    // the sheet prints cos(2w) - sin^4(w); differentiating y gives
    // cos(2w) - sin^2(w) (checked numerically), so we use that.
    dy:  b * eps * (Math.cos(2 * w) - s * s) / (D * D),
  };
}

// Rotation matrix built DIRECTLY from the tangent (tx, tz), no angles.
// Vehicles are modelled facing +X. We want:
//   R*(1,0,0) = f = (tx, 0, tz)     forward = unit tangent
//   R*(0,1,0) = u = (0, 1, 0)       up stays up
//   R*(0,0,1) = f x u = (-tz, 0, tx) side axis (right-handed)
// These three vectors are the columns of R (glm is column-major).
function headingMatrix(dx, dz) {
  const len = Math.hypot(dx, dz) || 1;
  const tx = dx / len, tz = dz / len;
  return glm.mat4( tx, 0, tz, 0,
                    0, 1,  0, 0,
                  -tz, 0, tx, 0,
                    0, 0,  0, 1);
}
</script>

## Exercise A.2 — Siege convoy

The convoy is formed by a **lead battering-ram vehicle** and **3 supply wagons**, all built from the primitives:

* every vehicle shares the same chassis: a deck (scaled cube), side boards, two axles (cylinders rotated to lie along
  $z$) and four wheels;
* **ram**: four posts, a sloped roof made of two rotated planks, a log (cylinder rotated to lie along $x$) with an iron
  cone tip, hanging from ropes. The ram group swings back and forth ($x = 0.3\sin 4t$);
* **wagon 1** — crates (cubes); **wagon 2** — barrels (open cylinders + disc lids); **wagon 3** — ammunition (a
  pyramid of stone spheres). The three wagons have different lengths, widths and wheel radii, and each has a tow bar
  linking it to the vehicle in front.

**Wheel toggle (key `R`).** Each wheel is a `Wheel` node. Its `update(spin, type)` method chooses the primitive and
the local transformation for that type:

| type | primitive | local transformation |
|---|---|---|
| cube | `Cube` | `S(0.8r, 0.8r, 0.18)` |
| cylinder | `Cylinder` | `R(90°, x) · S(r, 0.15, r)` (axis $y$ turned to $z$) |
| sphere | `Sphere` | `S(r, r, r)` |

and then `T(x, r, z) · R(-spin, z) · shapeMatrix`, so the wheel always touches the ground and **rolls**: the spin angle
is the travelled distance divided by the radius ($\theta = d/r$), with $d$ accumulated from the speed
$\lVert (\dot x, \dot y) \rVert$ of the path.

In [ ]:
%%html
<script>
// =====================================================================
//  EXERCISE A.2 - Siege convoy (compound objects)
// ---------------------------------------------------------------------
//  Each vehicle is a Node3D tree built from the basic primitives.
//  Vehicles are modelled facing +X, standing on the ground (y = 0).
//  `sh` is a palette of shared shape instances created once in the app.
// =====================================================================

// Wheel / roller: its geometry and local scale change with `type`
//   0 = cube, 1 = cylinder, 2 = sphere
window.Wheel = class Wheel extends Node3D {
  constructor(gl, sh, x, z, r) {
    super(gl);
    this.sh = sh; this.x = x; this.z = z; this.r = r;
    this.update(0, 1);
  }
  update(spin, type) {
    const r = this.r;
    let shape, shapeM;
    if (type === 0) {        // cube: square block of side ~1.6r
      shape = this.sh.wheelCube;   shapeM = S(0.8 * r, 0.8 * r, 0.18);
    } else if (type === 1) { // cylinder: axis y turned to z, then radius r
      shape = this.sh.wheelCyl;    shapeM = mul(R(90, 1, 0, 0), S(r, 0.15, r));
    } else {                 // sphere: a round roller ball of radius r
      shape = this.sh.wheelSphere; shapeM = S(r, r, r);
    }
    this.part = shape;
    // centre at (x, r, z) so it touches the ground; spin about z (rolling)
    this.localM = mul(T(this.x, r, this.z), R(-spin, 0, 0, 1), shapeM);
  }
};

// Common chassis for every vehicle: platform, side boards, 4 wheels
function makeChassis(gl, sh, len, wid, r) {
  const v = new Node3D(gl);
  const h = 2 * r + 0.1;                                   // platform height
  v.addPart(sh.wood, T(0, h, 0), S(len, 0.12, wid));      // deck
  v.addPart(sh.darkWood, T(0, h + 0.3, wid), S(len, 0.25, 0.06));   // side boards
  v.addPart(sh.darkWood, T(0, h + 0.3, -wid), S(len, 0.25, 0.06));
  // two axles (closed cylinders along z)
  for (const ax of [-0.65 * len, 0.65 * len])
    v.addPart(sh.ironCyl, T(ax, r, 0), R(90, 1, 0, 0), S(0.07, wid + 0.25, 0.07));
  v.wheels = [];
  for (const ax of [-0.65 * len, 0.65 * len])
    for (const sz of [-1, 1])
      v.wheels.push(v.add(new Wheel(gl, sh, ax, sz * (wid + 0.2), r)));
  v.deckY = h + 0.12;
  v.len = len;
  return v;
}

// Lead vehicle: covered battering ram
function makeRam(gl, sh) {
  const v = makeChassis(gl, sh, 1.8, 0.8, 0.45);
  const y = v.deckY;
  // four posts + two sloped roof planks (rotated about x)
  for (const px of [-1.5, 1.5]) for (const pz of [-0.7, 0.7])
    v.addPart(sh.darkWood, T(px, y + 0.8, pz), S(0.08, 0.8, 0.08));
  v.addPart(sh.roof, T(0, y + 1.75, 0.45), R(35, 1, 0, 0), S(1.9, 0.06, 0.6));
  v.addPart(sh.roof, T(0, y + 1.75, -0.45), R(-35, 1, 0, 0), S(1.9, 0.06, 0.6));
  // swinging ram: a group hung from the roof, animated in the app
  v.ram = v.add(new Node3D(gl));
  v.ram.addPart(sh.log, T(0, y + 0.75, 0), R(-90, 0, 0, 1), S(0.22, 1.9, 0.22)); // log along x
  v.ram.addPart(sh.ironCone, T(2.05, y + 0.75, 0), R(-90, 0, 0, 1), S(0.26, 0.25, 0.26)); // iron head (cone -> +x)
  for (const rx of [-1.0, 1.0])                                                 // ropes
    v.ram.addPart(sh.rope, T(rx, y + 1.15, 0), S(0.03, 0.4, 0.03));
  return v;
}

// Supply wagons, each with a different load profile and size
function makeWagon(gl, sh, kind) {
  const sizes = [[1.3, 0.75, 0.40], [1.1, 0.7, 0.35], [1.5, 0.8, 0.45]];
  const [len, wid, r] = sizes[kind];
  const v = makeChassis(gl, sh, len, wid, r);
  const y = v.deckY;
  // tow bar linking to the vehicle in front
  v.addPart(sh.ironCyl, T(len + 0.5, y - 0.1, 0), R(-90, 0, 0, 1), S(0.05, 0.5, 0.05));

  if (kind === 0) {          // crates of supplies
    for (const [cx, cz, cy] of [[-0.6, -0.35, 0], [0.6, -0.35, 0], [-0.6, 0.35, 0], [0.6, 0.35, 0], [0, 0, 0.6]])
      v.addPart(sh.crate, T(cx, y + 0.3 + cy, cz), S(0.3, 0.3, 0.3));
  } else if (kind === 1) {   // barrels with disc lids
    for (const bx of [-0.55, 0.05, 0.6]) {
      v.addPart(sh.barrel, T(bx, y + 0.45, 0), S(0.3, 0.45, 0.3));
      v.addPart(sh.lid, T(bx, y + 0.91, 0), S(0.28, 1, 0.28));
    }
  } else {                   // ammunition: stone balls in a small pyramid
    for (const bx of [-0.9, -0.3, 0.3, 0.9]) for (const bz of [-0.3, 0.3])
      v.addPart(sh.stone, T(bx, y + 0.3, bz), S(0.3, 0.3, 0.3));
    for (const bx of [-0.6, 0, 0.6])
      v.addPart(sh.stone, T(bx, y + 0.8, 0), S(0.3, 0.3, 0.3));
  }
  return v;
}
</script>

## Exercise A.3 — Movement along the lemniscate and rotation from the tangent

Each vehicle $i$ uses $\omega_i(t) = \varepsilon t + \vartheta_i$ with $\vartheta_i = -0.36\,i$ (wagons stay behind the
ram) and $\varepsilon = 0.22$, $a = b = 14$:

$$x_i = \frac{a\cos\omega_i}{1+\sin^2\omega_i}, \qquad y_i = \frac{b\sin\omega_i\cos\omega_i}{1+\sin^2\omega_i}.$$

The road lies on the ground, so the path coordinate $x$ is the world $X$ and the path coordinate $y$ is the world $Z$.

**Tangent.** Differentiating (with $s=\sin\omega$, $c=\cos\omega$, $D = 1+s^2$):

$$\frac{dx}{dt} = -a\varepsilon\, s\,\frac{2+c^2}{D^2}, \qquad
\frac{dy}{dt} = b\varepsilon\,\frac{\cos 2\omega - \sin^2\omega}{D^2}.$$

*Note:* the assignment sheet shows $\cos 2\omega - \sin^4\omega$ in $dy/dt$. Applying the quotient rule,
$\frac{d}{d\omega}\big(sc\big) = \cos 2\omega$, so the numerator is
$\cos 2\omega\,(1+s^2) - sc\cdot 2sc = \cos 2\omega + s^2(c^2 - s^2) - 2s^2c^2 = \cos 2\omega - s^2$.
We also checked it numerically against finite differences, so the code uses $\cos 2\omega - \sin^2\omega$.

**Rotation matrix from the tangent (no angles needed).** Every vehicle is modelled facing $+X$. Let
$\mathbf f = (t_x, 0, t_z)$ be the unit tangent ($t_x = \dot x/\lVert\dot{\mathbf p}\rVert$,
$t_z = \dot y/\lVert\dot{\mathbf p}\rVert$). We want a rotation that sends

* the model forward axis $(1,0,0)$ to $\mathbf f$,
* the model up axis $(0,1,0)$ to $\mathbf u = (0,1,0)$ (vehicles stay upright),
* the model side axis $(0,0,1)$ to $\mathbf f \times \mathbf u = (-t_z, 0, t_x)$ (keeps a right-handed frame).

Since $R\,\mathbf e_j$ is the $j$-th column of $R$, the images of the axes **are** the columns:

$$R = \begin{bmatrix} t_x & 0 & -t_z \\ 0 & 1 & 0 \\ t_z & 0 & t_x \end{bmatrix}.$$

The columns are unit length and mutually orthogonal and $\det R = t_x^2 + t_z^2 = 1$, so $R$ is a proper rotation.
(It equals a rotation about $y$ by $\theta$ with $\cos\theta = t_x$, $\sin\theta = -t_z$, but we never compute $\theta$.)
The model matrix of each vehicle is then $M_i = T(x_i, 0, y_i)\, R_i$ — see `headingMatrix` and `update()`.

## Exercises A.4 and A.5 — Camp props, animated objects and the castle

**A.4 — along the patrol road.** For 24 values of $\omega$ we take the road point and its normal
$\mathbf n = (-t_z, t_x)$, flip it to point out of the loop, and place a **torch** (pole + upside-down cone + flickering
sphere flame) or a **tree** (trunk + two cones) at 2.8 units from the road. Points near the crossing are skipped.

Two compound objects with local animation:

* **Trebuchet** — base beams, sheared uprights, axle and an arm with pendular motion
  $\theta(t) = 40^\circ \sin(1.3t)$. The counterweight hangs from a child node rotated by $-\theta$, so it stays
  vertical: the parent rotation and the child rotation cancel.
* **Windmill** — tower (cylinder), cone roof and four sails rotating continuously ($-50^\circ$ per second).

**A.5 — the castle** (front wall at $z=-14$):

* courtyard **floor**, four **walls** with battlements (rows of merlons), four round corner towers with cone roofs and two
  gatehouse towers;
* an **archway**: the opening is straight up to $y = 2.5$ and then a half circle of radius 2. The corners above the arch
  are filled with thin horizontal slabs whose half-width is $2-\sqrt{4-(y-2.5)^2}$, and 11 stone blocks rotated along
  the arc frame it;
* a **portcullis** (iron grid) that slides up/down smoothly with key `G` (it rises into the gatehouse block);
* **windows** (dark thin boxes) on the front wall, towers and the keep, which also has a door.

Repeated compound props: **wooden barricades** (row in front of the walls), **supply crate** stacks, **camp tables**
(with disc plates) and **weapon racks** (spears = cylinder + cone tip).

In [ ]:
%%html
<script>
// =====================================================================
//  EXERCISE A.4 - Camp props and animated compound objects
// =====================================================================

// Torch / brazier: pole + bowl (upside-down cone) + flickering flame
function makeTorch(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.darkWood, T(0, 0.9, 0), S(0.07, 0.9, 0.07));
  n.addPart(sh.ironCone, T(0, 1.95, 0), R(180, 1, 0, 0), S(0.28, 0.18, 0.28));
  n.flame = n.add(new Node3D(gl, sh.flame));   // scaled every frame in the app
  return n;
}

// Tree: trunk + two stacked cones of foliage
function makeTree(gl, sh, h = 1) {
  const n = new Node3D(gl, null, S(h, h, h));
  n.addPart(sh.trunk, T(0, 0.8, 0), S(0.18, 0.8, 0.18));
  n.addPart(sh.leaves, T(0, 2.1, 0), S(1.0, 0.8, 1.0));
  n.addPart(sh.leaves, T(0, 2.9, 0), S(0.7, 0.6, 0.7));
  return n;
}

// Trebuchet: frame + swinging arm (pendular motion). The counterweight
// hangs from the arm through its own node, which cancels the arm angle
// so the weight always stays vertical (hierarchy in action).
function makeTrebuchet(gl, sh) {
  const n = new Node3D(gl);
  for (const z of [-0.8, 0.8]) {
    n.addPart(sh.wood, T(0, 0.15, z), S(2.2, 0.15, 0.15));               // ground beams
    // leaning uprights: a SHEARED box (x' = x + k*y) keeps top and
    // bottom horizontal while the post leans
    n.addPart(sh.darkWood, T(-0.75, 1.65, z), SH(0.35), S(0.12, 1.5, 0.12));
    n.addPart(sh.darkWood, T(0.75, 1.65, z), SH(-0.35), S(0.12, 1.5, 0.12));
  }
  n.addPart(sh.ironCyl, T(0, 3.1, 0), R(90, 1, 0, 0), S(0.1, 1.0, 0.1));    // axle
  n.arm = n.add(new Node3D(gl));                   // rotated about z in the app
  n.arm.addPart(sh.log, T(1.0, 0, 0), R(-90, 0, 0, 1), S(0.1, 2.4, 0.1)); // arm from -1.4 to 3.4
  n.arm.addPart(sh.stone, T(3.4, 0, 0), S(0.25, 0.25, 0.25));            // projectile
  n.weight = n.arm.add(new Node3D(gl));            // pivot at the short end
  n.weight.addPart(sh.rope, T(0, -0.35, 0), S(0.03, 0.35, 0.03));
  n.weight.addPart(sh.crate, T(0, -0.95, 0), S(0.45, 0.35, 0.45));
  return n;
}

// Windmill: tower + roof + blades that rotate continuously
function makeWindmill(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.tower, T(0, 2.5, 0), S(1.3, 2.5, 1.3));
  n.addPart(sh.roof, T(0, 5.9, 0), S(1.6, 0.9, 1.6));
  n.addPart(sh.window, T(0, 1.0, 1.31), S(0.35, 0.6, 0.02));        // door
  n.addPart(sh.ironCyl, T(0, 4.6, 1.5), R(90, 1, 0, 0), S(0.12, 0.3, 0.12)); // hub
  n.blades = n.add(new Node3D(gl));                // rotated about z in the app
  for (let k = 0; k < 4; k++)
    n.blades.addPart(sh.sail, R(90 * k, 0, 0, 1), T(0, 1.6, 0), S(0.3, 1.4, 0.03));
  return n;
}

// =====================================================================
//  EXERCISE A.5 - Repeated camp props
// =====================================================================

// Wooden barricade: two crossed stakes + a horizontal log
function makeBarricade(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.log, T(0, 0.7, 0), R(-90, 0, 0, 1), S(0.1, 1.2, 0.1));
  for (const a of [40, -40])
    n.addPart(sh.darkWood, T(0, 0.6, 0), R(a, 0, 0, 1), S(0.08, 0.85, 0.08));
  return n;
}

// Stack of supply crates
function makeCrates(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.crate, T(-0.35, 0.3, 0), S(0.3, 0.3, 0.3));
  n.addPart(sh.crate, T(0.35, 0.3, 0.05), R(15, 0, 1, 0), S(0.3, 0.3, 0.3));
  n.addPart(sh.crate, T(0, 0.85, 0), R(-10, 0, 1, 0), S(0.25, 0.25, 0.25));
  return n;
}

// Camp table: top, four legs and two plates (discs)
function makeTable(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.wood, T(0, 0.8, 0), S(0.8, 0.05, 0.5));
  for (const x of [-0.7, 0.7]) for (const z of [-0.4, 0.4])
    n.addPart(sh.darkWood, T(x, 0.4, z), S(0.05, 0.4, 0.05));
  for (const x of [-0.4, 0.4])
    n.addPart(sh.plate, T(x, 0.86, 0), S(0.18, 1, 0.18));
  return n;
}

// Weapon rack: frame with three spears (cylinder + cone tip)
function makeRack(gl, sh) {
  const n = new Node3D(gl);
  n.addPart(sh.darkWood, T(0, 0.15, 0), S(0.8, 0.08, 0.2));
  n.addPart(sh.darkWood, T(0, 1.0, 0), S(0.8, 0.05, 0.05));
  for (const x of [-0.75, 0.75]) n.addPart(sh.darkWood, T(x, 0.6, 0), S(0.05, 0.6, 0.05));
  for (const x of [-0.45, 0, 0.45]) {
    n.addPart(sh.log, T(x, 0.95, 0), S(0.03, 0.85, 0.03));
    n.addPart(sh.ironCone, T(x, 1.9, 0), S(0.06, 0.12, 0.06));
  }
  return n;
}

// =====================================================================
//  EXERCISE A.5 - Castle: floor, walls, towers, archway with portcullis,
//  windows, battlements and a keep. Centred at the origin, gate facing +z.
// =====================================================================
function makeCastle(gl, sh) {
  const c = new Node3D(gl);
  const W = 12, D = 10, H = 5, th = 0.4;      // half width, half depth, wall height, half thickness

  c.addPart(sh.floor, T(0, 0.06, 0), S(W, 0.06, D));                     // courtyard floor

  // battlements: a row of merlons along a wall top
  const merlons = (x0, x1, z0, z1) => {
    const len = Math.hypot(x1 - x0, z1 - z0), n = Math.floor(len / 1.2);
    for (let i = 0; i <= n; i++) {
      const t = i / n;
      c.addPart(sh.wall, T(x0 + (x1 - x0) * t, H + 0.3, z0 + (z1 - z0) * t), S(0.3, 0.3, 0.45));
    }
  };

  // back and side walls (plain boxes)
  c.addPart(sh.wall, T(0, H / 2, -D), S(W, H / 2, th));   merlons(-W, W, -D, -D);
  c.addPart(sh.wall, T(-W, H / 2, 0), S(th, H / 2, D));   merlons(-W, -W, -D, D);
  c.addPart(sh.wall, T(W, H / 2, 0), S(th, H / 2, D));    merlons(W, W, -D, D);

  // front wall with the archway: opening 4 wide, straight up to y=2.5,
  // then a half-circle of radius 2 (apex at y=4.5)
  const gw = 2, ry = 2.5;
  for (const sgn of [-1, 1])                                          // wall either side of the gate
    c.addPart(sh.wall, T(sgn * (W + gw) / 2, H / 2, D), S((W - gw) / 2, H / 2, th));
  c.addPart(sh.wall, T(0, (4.5 + H) / 2, D), S(gw, (H - 4.5) / 2, th));  // above the arch
  // spandrels: thin horizontal slabs that fill the corners above the arch
  const step = 0.2;
  for (let y = ry; y < 4.5; y += step) {
    const yc = y + step / 2;
    const inner = Math.sqrt(Math.max(0, gw * gw - (yc - ry) ** 2));   // arch half-width at this height
    const half = (gw - inner) / 2;
    for (const sgn of [-1, 1])
      c.addPart(sh.wall, T(sgn * (gw - half), yc, D), S(half, step / 2, th));
  }
  // voussoirs: stone blocks following the arch, rotated along it
  for (let k = 0; k <= 10; k++) {
    const a = Math.PI * k / 10;
    c.addPart(sh.stoneTrim, T(2.15 * Math.cos(a), ry + 2.15 * Math.sin(a), D + 0.1),
              R(a * 180 / Math.PI, 0, 0, 1), S(0.18, 0.32, th + 0.05));
  }
  merlons(-W, -2, D, D); merlons(2, W, D, D);
  // gatehouse block above the gate: hides the raised portcullis
  c.addPart(sh.wall, T(0, H + 1.25, D), S(gw, 1.25, th + 0.1));
  for (const mx of [-1.5, 0, 1.5]) c.addPart(sh.wall, T(mx, H + 2.8, D), S(0.3, 0.3, 0.45));

  // portcullis: iron grid inside the arch, lifted by the app ('g' key)
  c.portcullis = c.add(new Node3D(gl));
  for (let x = -1.75; x <= 1.76; x += 0.5)
    c.portcullis.addPart(sh.ironBar, T(x, 2.25, D), S(0.05, 2.25, 0.05));
  for (let y = 0.5; y < 4.5; y += 0.8)
    c.portcullis.addPart(sh.ironBar, T(0, y, D), S(2, 0.05, 0.05));

  // gatehouse towers (boxes) and corner towers (cylinders + cone roofs)
  for (const sgn of [-1, 1]) {
    c.addPart(sh.wall, T(sgn * 3.5, 3.5, D + 0.4), S(1.5, 3.5, 1.2));
    c.addPart(sh.window, T(sgn * 3.5, 5.0, D + 1.62), S(0.25, 0.5, 0.02));
    for (const mx of [-1.2, 0, 1.2])
      c.addPart(sh.wall, T(sgn * 3.5 + mx, 7.3, D + 1.4), S(0.3, 0.3, 0.2));
  }
  for (const [x, z] of [[-W, -D], [W, -D], [-W, D], [W, D]]) {
    c.addPart(sh.tower, T(x, 3.5, z), S(1.6, 3.5, 1.6));
    c.addPart(sh.roof, T(x, 8.2, z), S(1.9, 1.2, 1.9));
    c.addPart(sh.window, T(x, 4.5, z + Math.sign(z) * 1.61), S(0.2, 0.45, 0.02));
  }
  // windows on the front wall (outside face)
  for (const x of [-9, -6, 6, 9])
    c.addPart(sh.window, T(x, 3.2, D + th + 0.02), S(0.3, 0.5, 0.02));

  // keep (main building) inside the courtyard, with windows and a door
  c.addPart(sh.wall, T(0, 3.5, -6), S(4, 3.5, 2.5));
  c.addPart(sh.roof, T(0, 8.0, -6), S(4.5, 1.0, 3.0));                     // cone roof
  c.addPart(sh.window, T(0, 1.2, -3.48), S(0.6, 1.2, 0.02));             // door
  for (const x of [-2.5, 2.5]) for (const y of [2.5, 5])
    c.addPart(sh.window, T(x, y, -3.48), S(0.35, 0.5, 0.02));
  return c;
}
</script>

## Exercises A.6 and A.7 — Cameras, and the application

**Visitor camera (A.6).** The camera keeps a position $(x, 1.7, z)$ (eye height of a scout), a yaw angle $\psi$ and a
pitch angle. The **horizontal** view direction is $\mathbf f = (\sin\psi, 0, \cos\psi)$ and the right vector is
$\mathbf r = \mathbf f \times \mathbf u = (-\cos\psi, 0, \sin\psi)$. Each frame, while a key is held, a displacement
is accumulated:  `W`: $+\mathbf f\,v\,\Delta t$, `S`: $-\mathbf f\,v\,\Delta t$, `D`: $+\mathbf r\,v\,\Delta t$,
`A`: $-\mathbf r\,v\,\Delta t$. Because $\mathbf f$ has no $y$ component, the scout walks on the ground even when looking
up or down. `Q/E` (or arrows) change the yaw/pitch. The view matrix is
`lookAt(eye, eye + d, up)` with $d = (\cos p\sin\psi, \sin p, \cos p\cos\psi)$.

**Ride-along camera (A.7, key `V`).** The eye is placed on the front edge of the rear (ammunition) wagon,
$\mathbf e = \mathbf p_3 + 1.0\,\mathbf f_3 + (0, 3.3, 0)$, and looks 10 units ahead along the same tangent
$\mathbf f_3$ used to orient the wagon, so the view follows the convoy and the fortress as the train goes around the
figure-eight.

**Keyboard.** The listeners are attached to the canvas (which gets `tabindex`), call `stopPropagation()` and
disable Jupyter's keyboard shortcuts while the canvas has focus, so pressing `A`, `S`, `D` or `V` does not edit the
notebook. Pressed keys are kept in a set, which gives smooth continuous movement.

Each frame `render()` computes $\Delta t$, updates all animations (`update`), builds
$P$ = `perspective(60°, aspect, 0.1, 300)` and the view matrix $V$, and draws the whole tree with $PV$.

In [ ]:
%%html
<canvas id="siegeCanvas" width="900" height="560" style="border:2px solid #000000;">
      Error: Your browser does not support the HTML canvas tag.
</canvas>
<div id="siegeHud" style="font-family:monospace"></div>

<script>
// =====================================================================
//  The application: builds the scene, animates it, handles the keys
//  and the two cameras (Exercises A.2 - A.7)
// =====================================================================

// Path parameters (Exercise A.3)
var PATH = { a: 14, b: 14, eps: 0.22, gap: 0.36 };   // gap = theta spacing between vehicles

// colour helper: c1 -> c2 gradient for every shape
var col = {
  green: [[0.35, 0.6, 0.25], [0.25, 0.45, 0.18]], dirt: [[0.6, 0.48, 0.3], [0.5, 0.38, 0.22]],
  wood: [[0.62, 0.42, 0.22], [0.45, 0.28, 0.12]], dark: [[0.4, 0.26, 0.13], [0.28, 0.17, 0.08]],
  stone: [[0.72, 0.7, 0.66], [0.52, 0.5, 0.48]], red: [[0.75, 0.2, 0.15], [0.5, 0.1, 0.08]],
  iron: [[0.45, 0.45, 0.5], [0.25, 0.25, 0.3]], fire: [[1.0, 0.85, 0.2], [0.95, 0.3, 0.05]],
  leaf: [[0.2, 0.55, 0.2], [0.08, 0.35, 0.1]], black: [[0.12, 0.1, 0.1], [0.05, 0.05, 0.08]],
  sail: [[0.95, 0.93, 0.85], [0.8, 0.78, 0.7]], rope: [[0.8, 0.7, 0.5], [0.7, 0.6, 0.4]],
};

window.SiegeApp = class SiegeApp extends DEECapp {
  initialize() {
    // shaders from the cells above (same as 04Objects)
    this.shaderprog = new DEECshader(this.gl);
    this.shaderprog.srcShaders(document.getElementById("my-vertex-shader").text,
                               document.getElementById("my-fragment-shader").text);
    this.gl.enable(this.gl.DEPTH_TEST);
    this.gl.clearColor(0.6, 0.78, 0.95, 1.0);              // sky

    this.makePalette();
    this.buildScene();

    // state
    this.time = 0; this.convoyTime = 0; this.last = performance.now();
    this.paused = false;
    this.wheelType = 1;                  // 0 cube, 1 cylinder, 2 sphere
    this.camMode = "visitor";            // or "ride"
    this.cam = { x: 0, z: 22, yaw: Math.PI, pitch: -0.08 };   // looking towards -z (the castle)
    this.gateOpen = false; this.gateLift = 0;
    this.setupKeys();
  }

  // One instance of each coloured primitive, re-used by many nodes
  makePalette() {
    const gl = this.gl, sh = {};
    const reg = (name, shape) => { shape.setShader(this.shaderprog); sh[name] = shape; };
    reg("ground", new Cube(gl, ...col.green));     reg("road", new Cube(gl, ...col.dirt));
    reg("wood", new Cube(gl, ...col.wood));        reg("darkWood", new Cube(gl, ...col.dark));
    reg("crate", new Cube(gl, ...col.wood));       reg("wall", new Cube(gl, ...col.stone));
    reg("floor", new Cube(gl, ...col.stone));      reg("stoneTrim", new Cube(gl, ...col.iron));
    reg("window", new Cube(gl, ...col.black));     reg("sail", new Cube(gl, ...col.sail));
    reg("ironBar", new Cube(gl, ...col.iron));
    reg("roof", new Cone(gl, ...col.red));         reg("leaves", new Cone(gl, ...col.leaf));
    reg("ironCone", new Cone(gl, ...col.iron));
    reg("log", new Cylinder(gl, ...col.wood));     reg("trunk", new Cylinder(gl, ...col.dark));
    reg("tower", new Cylinder(gl, ...col.stone));  reg("ironCyl", new Cylinder(gl, ...col.iron));
    reg("rope", new Cylinder(gl, ...col.rope));
    reg("barrel", new Cylinder(gl, ...col.dark, false));   // OPEN cylinder, closed by a disc lid
    reg("lid", new Disc(gl, ...col.wood));         reg("plate", new Disc(gl, ...col.iron));
    reg("stone", new Sphere(gl, ...col.iron));     reg("flame", new Sphere(gl, ...col.fire));
    reg("wheelCube", new Cube(gl, ...col.dark));   reg("wheelCyl", new Cylinder(gl, ...col.dark));
    reg("wheelSphere", new Sphere(gl, ...col.dark));
    this.sh = sh;
  }

  buildScene() {
    const gl = this.gl, sh = this.sh;
    const world = this.world = new Node3D(gl);
    world.addPart(sh.ground, T(0, -0.05, -5), S(60, 0.05, 60));

    // --- road along the figure-eight (small boxes aligned with the tangent)
    const N = 160;
    for (let k = 0; k < N; k++) {
      const p = lemniscate(0, PATH.a, PATH.b, 1, 2 * Math.PI * k / N);
      const q = lemniscate(0, PATH.a, PATH.b, 1, 2 * Math.PI * (k + 1) / N);
      const len = Math.hypot(q.x - p.x, q.y - p.y);
      world.addPart(sh.road, T((p.x + q.x) / 2, 0.02, (p.y + q.y) / 2),
                    headingMatrix(q.x - p.x, q.y - p.y), S(len / 2 + 0.05, 0.02, 1.3));
    }

    // --- convoy: ram + 3 wagons (A.2)
    this.vehicles = [makeRam(gl, sh), makeWagon(gl, sh, 0), makeWagon(gl, sh, 1), makeWagon(gl, sh, 2)];
    for (const v of this.vehicles) { world.add(v); v.dist = 0; }

    // --- torches and trees around the patrol path (A.4)
    this.torches = [];
    for (let k = 0; k < 24; k++) {
      const w = 2 * Math.PI * k / 24;
      if (Math.abs(Math.cos(w)) < 0.35) continue;         // keep the crossing clear
      const p = lemniscate(0, PATH.a, PATH.b, 1, w);
      const len = Math.hypot(p.dx, p.dy);
      let nx = -p.dy / len, nz = p.dx / len;               // normal to the road
      const lobe = Math.sign(p.x) * 8.5;                   // centre of this loop
      if (nx * (p.x - lobe) + nz * p.y < 0) { nx = -nx; nz = -nz; }   // point outwards
      const pos = T(p.x + 2.8 * nx, 0, p.y + 2.8 * nz);
      if (k % 2 === 0) {
        const t = world.add(makeTorch(gl, sh)); t.localM = pos; this.torches.push(t);
      } else {
        const tr = world.add(makeTree(gl, sh, 0.8 + 0.3 * ((k * 7) % 3) / 2)); tr.localM = pos;
      }
    }
    for (const [x, z, s] of [[-8.5, 0, 1.1], [8.5, 0.5, 1.2], [-25, -10, 1.4], [25, 10, 1.3],
                             [-26, 6, 1.0], [18, -22, 1.5], [-18, -24, 1.4], [27, -12, 1.1]]) {
      const tr = world.add(makeTree(gl, sh, s)); tr.localM = mul(T(x, 0, z), S(s, s, s));
    }

    // --- animated compound objects (A.4)
    this.trebuchet = world.add(makeTrebuchet(gl, sh));
    this.trebuchet.localM = mul(T(-10, 0, 11), R(90, 0, 1, 0));     // throws towards the castle
    this.windmill = world.add(makeWindmill(gl, sh));
    this.windmill.localM = T(22, 0, -2);

    // --- castle (A.5): its front wall is at z = -14
    this.castle = world.add(makeCastle(gl, sh));
    this.castle.localM = T(0, 0, -24);

    // --- repeated props (A.5)
    for (let x = -12; x <= 12; x += 2.6) {
      if (Math.abs(x) < 3.5) continue;                       // leave the gate road open
      const b = world.add(makeBarricade(gl, sh)); b.localM = T(x, 0, -9.5);
    }
    for (const [x, z, a] of [[-14, 9, 0], [-6, 12, 30], [3, -26, 0], [-7, -21, 20], [15, 7, -20], [7, -30, 0]]) {
      const c = world.add(makeCrates(gl, sh)); c.localM = mul(T(x, 0, z), R(a, 0, 1, 0));
    }
    for (const [x, z, a] of [[7, 14.5, 0], [10.5, 16, 25], [14, 14, -15], [-5, -19, 90]]) {
      const t = world.add(makeTable(gl, sh)); t.localM = mul(T(x, 0, z), R(a, 0, 1, 0));
    }
    for (const [x, z, a] of [[-15, 12, 0], [-3, 13, 10], [9, -17, 0], [-9, -17, 0]]) {
      const r = world.add(makeRack(gl, sh)); r.localM = mul(T(x, 0, z), R(a, 0, 1, 0));
    }
  }

  // ---------------- keyboard ----------------
  setupKeys() {
    this.keys = new Set();
    const cv = this.canvas || document.getElementById("siegeCanvas");
    cv.setAttribute("tabindex", "0");               // canvas can take keyboard focus
    // While the canvas has focus, Jupyter's own shortcuts (a, s, d, v...) are disabled
    cv.addEventListener("focus", () => { if (window.Jupyter) Jupyter.keyboard_manager.disable(); });
    cv.addEventListener("blur", () => { this.keys.clear(); if (window.Jupyter) Jupyter.keyboard_manager.enable(); });
    cv.addEventListener("click", () => cv.focus());
    cv.addEventListener("keydown", (e) => {
      e.stopPropagation(); e.preventDefault();
      const k = e.key.toLowerCase();
      this.keys.add(k);
      if (e.repeat) return;
      if (k === "v") this.camMode = this.camMode === "visitor" ? "ride" : "visitor";
      if (k === "r") this.wheelType = (this.wheelType + 1) % 3;
      if (k === "g") this.gateOpen = !this.gateOpen;
      if (k === "p") this.paused = !this.paused;
    });
    cv.addEventListener("keyup", (e) => { e.stopPropagation(); this.keys.delete(e.key.toLowerCase()); });
  }

  // ---------------- per-frame update ----------------
  update(dt) {
    this.time += dt;
    if (!this.paused) this.convoyTime += dt;
    const t = this.time;

    // convoy on the lemniscate (A.3): position + rotation from the tangent
    this.vehicles.forEach((v, i) => {
      const p = lemniscate(this.convoyTime, PATH.a, PATH.b, PATH.eps, -i * PATH.gap);
      v.localM = mul(T(p.x, 0, p.y), headingMatrix(p.dx, p.dy));
      v.tangent = [p.dx, p.dy]; v.pos = [p.x, p.y];
      if (!this.paused) v.dist += Math.hypot(p.dx, p.dy) * dt;   // distance travelled
      for (const w of v.wheels)                                 // rolling: angle = distance / radius
        w.update(v.dist / w.r * 180 / Math.PI, this.wheelType);
    });
    const ram = this.vehicles[0].ram;
    ram.localM = T(0.3 * Math.sin(4 * t), 0, 0);                // ram swings back and forth

    // torches flicker
    this.torches.forEach((tc, k) =>
      tc.flame.localM = mul(T(0, 2.25, 0), S(0.2, 0.28 * (1 + 0.25 * Math.sin(9 * t + k)), 0.2)));

    // trebuchet: pendular arm; the counterweight cancels the angle to hang straight down
    const th = 40 * Math.sin(1.3 * t);
    this.trebuchet.arm.localM = mul(T(0, 3.1, 0), R(th, 0, 0, 1));
    this.trebuchet.weight.localM = mul(T(-1.4, 0, 0), R(-th, 0, 0, 1));

    // windmill blades rotate continuously
    this.windmill.blades.localM = mul(T(0, 4.6, 1.75), R(-50 * t, 0, 0, 1));

    // portcullis slides smoothly to its target height
    const target = this.gateOpen ? 2.8 : 0;
    this.gateLift += (target - this.gateLift) * Math.min(1, 2 * dt);
    this.castle.portcullis.localM = T(0, this.gateLift, 0);

    // visitor camera (A.6): move along the horizontal view direction
    const c = this.cam, speed = 6 * dt, turn = 1.6 * dt;
    if (this.keys.has("arrowleft") || this.keys.has("q")) c.yaw += turn;
    if (this.keys.has("arrowright") || this.keys.has("e")) c.yaw -= turn;
    if (this.keys.has("arrowup")) c.pitch = Math.min(1.2, c.pitch + turn);
    if (this.keys.has("arrowdown")) c.pitch = Math.max(-1.2, c.pitch - turn);
    const fx = Math.sin(c.yaw), fz = Math.cos(c.yaw);       // forward (horizontal)
    const rx = -fz, rz = fx;                                // right = forward x up
    if (this.keys.has("w")) { c.x += fx * speed; c.z += fz * speed; }
    if (this.keys.has("s")) { c.x -= fx * speed; c.z -= fz * speed; }
    if (this.keys.has("d")) { c.x += rx * speed; c.z += rz * speed; }
    if (this.keys.has("a")) { c.x -= rx * speed; c.z -= rz * speed; }
  }

  // View matrix of the active camera
  viewMatrix() {
    const up = glm.vec3(0, 1, 0);
    if (this.camMode === "visitor") {
      const c = this.cam;
      const eye = glm.vec3(c.x, 1.7, c.z);                 // eye height of a scout
      const dir = glm.vec3(Math.cos(c.pitch) * Math.sin(c.yaw), Math.sin(c.pitch),
                           Math.cos(c.pitch) * Math.cos(c.yaw));
      return glm.lookAt(eye, eye['+'](dir), up);
    }
    // ride-along camera (A.7): on top of the rear wagon, looking ahead
    const v = this.vehicles[3];
    const len = Math.hypot(v.tangent[0], v.tangent[1]);
    const fx = v.tangent[0] / len, fz = v.tangent[1] / len;
    const eye = glm.vec3(v.pos[0] + 1.0 * fx, 3.3, v.pos[1] + 1.0 * fz);   // front edge of the wagon
    return glm.lookAt(eye, glm.vec3(eye.x + 10 * fx, 1.8, eye.z + 10 * fz), up);
  }

  render() {
    const now = performance.now();
    const dt = Math.min(0.05, (now - this.last) / 1000);
    this.last = now;
    this.update(dt);

    this.gl.clear(this.gl.COLOR_BUFFER_BIT | this.gl.DEPTH_BUFFER_BIT);
    const cv = this.gl.canvas;
    const P = glm.perspective(glm.radians(60), cv.width / cv.height, 0.1, 300);
    const PV = P['*'](this.viewMatrix());
    this.world.drawTree(PV, glm.mat4(1));

    const hud = document.getElementById("siegeHud");
    if (hud) hud.textContent = `camera: ${this.camMode} | wheels: ${["cubes", "cylinders", "spheres"][this.wheelType]}` +
      `${this.paused ? " | convoy paused" : ""}  —  click the canvas, then W A S D move, Q/E or arrows look, ` +
      `V camera, R wheels, G gate, P pause`;
  }
};

window.app = new SiegeApp("siegeCanvas");
app.run();
</script>

## Final notes

* Everything is WebGL 1.0 with the custom shaders above; all geometry is generated by our functions in the A.1 cell.
* Every class (`Shape`, the five primitives, `Node3D`, `Wheel`) inherits from `CGRAobject`; the application inherits
  from `DEECapp`, as in the course example.
* Each primitive is created **once** per colour (the "palette" in `makePalette`) and re-used by many nodes: before each
  draw the node sets the matrix and calls `drawit()`, which saves GPU memory.